# AlzDetect — v3 "Medical-Safe" Training

Run 2 (full pool + file-repetition oversampling) is the true **68%** baseline.
Run 3 (heavy aug + class_weight + fine-tuning) dropped to **66%**.

This notebook implements the medical-safe plan:
1. **Medical-safe augmentation only** — ≤5° rotation + horizontal flip, nothing else.
2. **SMOTE in feature space** instead of `class_weight` (keeps CNN gradients clean).
3. **Frozen MobileNetV2 + tiny Dense(32, swish) adapter** (fast train, no catastrophic forgetting).

Exports a single drop-in `alz_mobilenetv2.keras`.

**Runtime:** GPU recommended (Runtime → Change runtime type → GPU).

## 1. Install dependencies

In [ ]:
!pip install -q kaggle scikit-fuzzy imbalanced-learn opencv-python-headless

## 2. Kaggle credentials
Paste your Kaggle username + API key (from kaggle.com → Settings → Create New Token).

In [ ]:
import os
os.environ['KAGGLE_USERNAME'] = ''  # <-- your kaggle username
os.environ['KAGGLE_KEY']      = ''  # <-- your kaggle API key
assert os.environ['KAGGLE_USERNAME'] and os.environ['KAGGLE_KEY'], 'Fill in your Kaggle credentials above.'

## 3. Download the mirror dataset
The original `sachinkumar413/alzheimer-mri-dataset` was removed from Kaggle; we use the `legendahmed/alzheimermridataset` mirror, which has a flat `all image/` folder (5121 images, class encoded in the filename prefix).

In [ ]:
!kaggle datasets download -d legendahmed/alzheimermridataset -p data --unzip
DATA_DIR = 'data/Alzheimer_s Dataset/all image'
import os
print('Files found:', len(os.listdir(DATA_DIR)))

## 4. Configuration & helpers

In [ ]:
import os, re
import numpy as np

IMG_SIZE = 128
EPOCHS   = 30
BATCH    = 32
SEED     = 42
np.random.seed(SEED)

PREFIX_TO_CLASS = {
    'mildDem': 'Mild Dementia',
    'moderateDem': 'Moderate Dementia',
    'verymildDem': 'Very mild Dementia',
    'nonDem': 'Non Demented',
}
CLASS_ORDER = ['Mild Dementia', 'Moderate Dementia', 'Very mild Dementia', 'Non Demented']


def fuzzy_targets(counts):
    """skfuzzy Mamdani controller: imbalance -> resample_factor."""
    import skfuzzy as fuzz
    import skfuzzy.control as ctrl
    imbalance = ctrl.Antecedent(np.arange(0, 1.1, 0.1), 'imbalance')
    resample  = ctrl.Consequent(np.arange(0, 1.1, 0.1), 'resample_factor')
    imbalance['low']    = fuzz.trimf(imbalance.universe, [0, 0, 0.5])
    imbalance['medium'] = fuzz.trimf(imbalance.universe, [0.3, 0.5, 0.7])
    imbalance['high']   = fuzz.trimf(imbalance.universe, [0.5, 1, 1])
    resample['low']     = fuzz.trimf(resample.universe, [0, 0, 0.3])
    resample['medium']  = fuzz.trimf(resample.universe, [0.2, 0.5, 0.7])
    resample['high']    = fuzz.trimf(resample.universe, [0.6, 1, 1])
    sim = ctrl.ControlSystemSimulation(ctrl.ControlSystem([
        ctrl.Rule(imbalance['low'], resample['low']),
        ctrl.Rule(imbalance['medium'], resample['medium']),
        ctrl.Rule(imbalance['high'], resample['high']),
    ]))
    nmax = max(counts.values())
    out = {}
    for c, n in counts.items():
        sim.input['imbalance'] = 1 - (n / nmax)
        sim.compute()
        out[c] = int(n + sim.output['resample_factor'] * nmax)
    return out


def bucket_files():
    buckets = {c: [] for c in CLASS_ORDER}
    for fn in sorted(os.listdir(DATA_DIR)):
        m = re.match(r'^[a-zA-Z]+', fn)
        if m and PREFIX_TO_CLASS.get(m.group()):
            buckets[PREFIX_TO_CLASS[m.group()]].append(fn)
    return buckets


def medical_safe_aug(img, rng):
    """<=5 deg rotation + optional horizontal flip. Nothing else."""
    import cv2
    angle = rng.uniform(-5, 5)
    h, w = img.shape[:2]
    M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
    img = cv2.warpAffine(img, M, (w, h), borderMode=cv2.BORDER_REFLECT)
    if rng.random() < 0.5:
        img = cv2.flip(img, 1)
    return img

## 5. Load images + medical-safe oversampling to the fuzzy targets

In [ ]:
import cv2

rng = np.random.default_rng(SEED)
buckets = bucket_files()
counts  = {c: len(f) for c, f in buckets.items()}
print('Real counts:', counts)

targets = fuzzy_targets(counts)
print('Fuzzy target counts:', targets)

class_map = {label: i for i, label in enumerate(CLASS_ORDER)}
data, labels = [], []
for label in CLASS_ORDER:
    files, target = buckets[label], targets[label]
    imgs = []
    for fn in files:
        im = cv2.imread(os.path.join(DATA_DIR, fn))
        if im is not None:
            imgs.append(cv2.resize(im, (IMG_SIZE, IMG_SIZE)))
    for im in imgs:
        data.append(im); labels.append(class_map[label])
    i = 0
    while imgs and sum(1 for l in labels if l == class_map[label]) < target:
        data.append(medical_safe_aug(imgs[i % len(imgs)], rng))
        labels.append(class_map[label]); i += 1

data  = np.array(data, dtype=np.float32)  # RAW 0-255 (matches backend/model.py)
y_int = np.array(labels)
print('After safe aug:', {CLASS_ORDER[i]: int((y_int == i).sum()) for i in range(len(CLASS_ORDER))})

## 6. Frozen MobileNetV2 → extract features

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D, Input
from tensorflow.keras.models import Model
from sklearn.model_selection import train_test_split

Xtr_img, Xte_img, ytr_int, yte_int = train_test_split(
    data, y_int, test_size=0.2, stratify=y_int, random_state=SEED)

inp  = Input(shape=(IMG_SIZE, IMG_SIZE, 3))
base = MobileNetV2(weights='imagenet', include_top=False, input_tensor=inp)
base.trainable = False
pooled = GlobalAveragePooling2D()(base.output)
feature_model = Model(inp, pooled)

print('Extracting frozen features ...')
Ftr = feature_model.predict(Xtr_img, batch_size=BATCH, verbose=1)
Fte = feature_model.predict(Xte_img, batch_size=BATCH, verbose=1)

## 7. SMOTE in feature space (train only)

In [ ]:
from imblearn.over_sampling import SMOTE
from tensorflow.keras.utils import to_categorical

min_class = min(int((ytr_int == i).sum()) for i in range(len(CLASS_ORDER)))
k = max(1, min(5, min_class - 1))
print(f'SMOTE k_neighbors={k} (smallest train class={min_class})')
Ftr_bal, ytr_bal = SMOTE(random_state=SEED, k_neighbors=k).fit_resample(Ftr, ytr_int)
print('Balanced:', {CLASS_ORDER[i]: int((ytr_bal == i).sum()) for i in range(len(CLASS_ORDER))})

ytr_cat = to_categorical(ytr_bal, num_classes=len(CLASS_ORDER))
yte_cat = to_categorical(yte_int, num_classes=len(CLASS_ORDER))

## 8. Train the tiny adapter head

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import classification_report

feat_in = Input(shape=(Ftr.shape[1],))
h = Dense(32, activation='swish')(feat_in)
h = Dropout(0.3)(h)
head_out = Dense(len(CLASS_ORDER), activation='softmax')(h)
head = Model(feat_in, head_out)
head.compile(optimizer=Adam(1e-3), loss='categorical_crossentropy', metrics=['accuracy'])

head.fit(Ftr_bal, ytr_cat, validation_data=(Fte, yte_cat),
         epochs=EPOCHS, batch_size=BATCH,
         callbacks=[EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True)])

yp = np.argmax(head.predict(Fte), axis=1)
print('\nClassification report:')
print(classification_report(yte_int, yp, target_names=CLASS_ORDER))

## 9. Stitch into one drop-in model + save
Combines the frozen base and trained head into a single end-to-end `alz_mobilenetv2.keras` — no changes needed in `backend/model.py`.

In [ ]:
full = Model(inp, head(feature_model(inp)))
full.save('alz_mobilenetv2.keras')
print('Saved alz_mobilenetv2.keras')

from google.colab import files  # download to send back / host for MODEL_URL
files.download('alz_mobilenetv2.keras')